In [ ]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import igraph as ig
from matplotlib.patches import Patch

# ---------------------------------------------------------------------
# formatted analysis toolbox
# ---------------------------------------------------------------------
sys.path.insert(0, "/home/ana/MicroBrain/codes/Graph Analysis & by region/Graph analysis")
from graph_analysis_functions_formatted import *

# ---------------------------------------------------------------------
# paths / params
# ---------------------------------------------------------------------
out_root = "/home/admin/Ana/MicroBrain/output/SMC_FULL_ANALYSIS_FORMATTED"
os.makedirs(out_root, exist_ok=True)

PATHS = {
    "SMC_1": "/home/admin/Ana/MicroBrain/output/um_gaia/formatted/formatted2/graph_18_OutGeom_um_formatted_Scut1.pkl",
    "SMC_2": "/home/admin/Ana/MicroBrain/output/um_gaia/formatted/formatted2/graph_18_OutGeom_um_formatted_Scut2.pkl",
    "SMC_3": "/home/admin/Ana/MicroBrain/output/um_gaia/formatted/formatted2/graph_18_OutGeom_um_formatted_Scut3.pkl",
}

HIPPO_CENTERS = [
    [1050, 1200, 1200],
    [1100, 1300,  1200],
    [700, 2000,  1500],
]

box_size_um = (400, 400, 400)

degree_thr = 4
eps_vox    = 2.0

# Density settings
slab_um_cutbox = 10.0
slab_axis      = "z"

# plotting controls
plot_paths_cap = 15
bins_hist = 40

BOX_ORDER = list(PATHS.keys())

V_COL = {
    "arteriole": "#d62728",
    "venule":    "#1f77b4",
    "capillary": "#7f7f7f",
    "unknown":   "#c7c7c7",
}
BOX_COL = {
    "SMC_1": "tab:blue",
    "SMC_2": "tab:orange",
    "SMC_3": "tab:green",
}

FACE_GRID = [
    ["y_max", "z_max", "y_min"],
    ["x_min", "z_min", "x_max"],
]

# ---------------------------------------------------------------------
# helpers (same as yours)
# ---------------------------------------------------------------------
def safe_arr(x, dtype=float):
    return np.asarray(x, dtype=dtype)

def safe_finite(x):
    x = safe_arr(x, float)
    return x[np.isfinite(x)]

def ecdf_xy(x):
    x = safe_finite(x)
    if x.size == 0:
        return np.array([]), np.array([])
    x = np.sort(x)
    y = np.arange(1, x.size + 1) / x.size
    return x, y

def plot_boxplot_by_graph(df_long, value_col, title, ylabel,  graphs_order=("SMC_1","SMC_2","SMC_3")):
    groups, labels = [], []
    for g in graphs_order:
        if g not in df_long["graph"].unique():
            continue
        x = df_long.loc[df_long["graph"] == g, value_col].to_numpy(float)
        x = x[np.isfinite(x)]
        groups.append(x)
        labels.append(g)

    plt.figure(figsize=(7, 5))
    plt.boxplot(groups, labels=labels, showfliers=False)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.grid(alpha=0.25, axis="y")
    plt.tight_layout()
    plt.show()

def plot_hist_overlay(df_long, value_col, title, xlabel, bins=40, graphs_order=("SMC_1","SMC_2","SMC_3")):
    plt.figure(figsize=(8, 5))
    for g in graphs_order:
        if g not in df_long["graph"].unique():
            continue
        x = df_long.loc[df_long["graph"] == g, value_col].to_numpy(float)
        x = x[np.isfinite(x)]
        if x.size:
            plt.hist(x, bins=bins, density=True, alpha=0.45, label=g)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Density")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

def diameter_length_overlay_by_type(dl, bins=40, graphs_order=("SMC_1","SMC_2","SMC_3"), box_label=""):
    for t in sorted(dl["type"].unique()):
        sub = dl[dl["type"] == t].copy()
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        fig.suptitle(f"{t} | Diameter & Length distributions | {box_label}", fontsize=12, fontweight="bold")

        ax = axes[0]
        for g in graphs_order:
            x = sub.loc[sub["graph"] == g, "diameter_um"].to_numpy(float)
            x = x[np.isfinite(x)]
            if x.size:
                ax.hist(x, bins=bins, density=True, alpha=0.45, label=g)
        ax.set_title("Diameter")
        ax.set_xlabel("µm")
        ax.set_ylabel("Density")
        ax.grid(alpha=0.25)
        ax.legend()

        ax = axes[1]
        for g in graphs_order:
            x = sub.loc[sub["graph"] == g, "length_um"].to_numpy(float)
            x = x[np.isfinite(x)]
            if x.size:
                ax.hist(x, bins=bins, density=True, alpha=0.45, label=g)
        ax.set_title("Length")
        ax.set_xlabel("Diameter (atlas-proxy, µm units)")
        ax.set_ylabel("Density")
        ax.grid(alpha=0.25)
        ax.legend()

        plt.tight_layout()
        plt.show()

def load_formatted_graph(path: str) -> ig.Graph:
    with open(path, "rb") as f:
        obj = pickle.load(f)
    if isinstance(obj, ig.Graph):
        return obj
    if isinstance(obj, dict) and "graph" in obj and isinstance(obj["graph"], ig.Graph):
        return obj["graph"]
    raise TypeError(f"File does not contain an igraph.Graph: {path} (type={type(obj)})")

def keep_giant_component(G: ig.Graph) -> ig.Graph:
    comps = G.components()
    if len(comps) <= 1:
        return G
    keep = np.asarray(comps[np.argmax(comps.sizes())], dtype=int)
    return G.induced_subgraph(keep)

def plot_intra_box_heterogeneity(df_box, box_name, slab_um):
    """Plots the density variation within a single box to spot errors."""
    if df_box is None or df_box.empty:
        return

    y_values = df_box["total_vol_frac"].to_numpy(float)
    x_mid = 0.5 * (df_box["slab_lo"].values + df_box["slab_hi"].values)

    plt.figure(figsize=(8, 4))
    plt.plot(x_mid, y_values, marker='o', color='firebrick', linewidth=2)
    plt.fill_between(x_mid, y_values, alpha=0.1, color='firebrick')

    # Add a horizontal line for the mean to see how much it fluctuates
    plt.axhline(np.nanmean(y_values), color='black', linestyle='--', alpha=0.5,
                label=f'Mean: {np.nanmean(y_values):.2f}%')

    plt.title(f"Intra-Box Density Profile ({slab_um}µm slabs) | {box_name}")
    plt.xlabel(f"Depth along {slab_axis}-axis (µm)")
    plt.ylabel("Vessel Volume Fraction (%)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_grouped_boxplot_types_per_graph(
    df_long,
    value_col="diameter_um",
    type_col="type",
     graphs_order=("SMC_1","SMC_2","SMC_3"),
    types_order=("arteriole","venule","capillary"),
    type_colors=None,
    title="Diameter by vessel type within each box",
    ylabel="Diameter (stored units)"
):
    """
    One figure:
      x-axis = graphs  ("SMC_1","SMC_2","SMC_3")
      For each graph, 3 boxplots (arteriole/venule/capillary) side-by-side.
    """
    if df_long is None or df_long.empty:
        print("[grouped boxplot] df_long is empty -> nothing to plot.")
        return

    if type_colors is None:
        type_colors = {"arteriole":"red","venule":"blue","capillary":"gray"}

    df = df_long.copy()
    df[type_col] = df[type_col].astype(str).str.lower().str.strip()
    df[type_col] = df[type_col].replace({
        "artery": "arteriole",
        "arterial": "arteriole",
        "vein": "venule",
        "venous": "venule",
        "cap": "capillary",
    })

    fig, ax = plt.subplots(figsize=(9, 5))
    base = np.arange(len(graphs_order), dtype=float)

    offsets = {types_order[0]: -0.28, types_order[1]: 0.00, types_order[2]: 0.28}
    width = 0.22

    for t in types_order:
        data, pos = [], []
        for i, g in enumerate(graphs_order):
            x = df.loc[(df["graph"] == g) & (df[type_col] == t), value_col].to_numpy(float)
            x = x[np.isfinite(x)]
            if x.size == 0:
                continue
            data.append(x)
            pos.append(base[i] + offsets[t])

        if not data:
            continue

        bp = ax.boxplot(
            data,
            positions=pos,
            widths=width,
            patch_artist=True,
            showfliers=False
        )
        for patch in bp["boxes"]:
            patch.set_facecolor(type_colors.get(t, "lightgray"))
            patch.set_alpha(0.70)
        for k in ["whiskers", "caps", "medians"]:
            for line in bp[k]:
                line.set_color("black")

    ax.set_xticks(base)
    ax.set_xticklabels(list(graphs_order))
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25, axis="y")

    handles = [Patch(facecolor=type_colors.get(t, "lightgray"), edgecolor="black", label=t) for t in types_order]
    ax.legend(handles=handles, title="Vessel type", frameon=False)

    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------------
# ADDED: Density helper (convert radii vox -> µm ONLY for density)
# ---------------------------------------------------------------------
def microsegments_for_density_um(ms, G, res_um_per_vox):
    """
    Returns a copy of ms where r0/r1 are in µm ONLY if graph stores diameter in vox.
    Assumes ms["lengths"] and ms["midpoints"] are already in µm (since you cut box in µm).
    """
    ms2 = dict(ms)

    # decide diameter unit from graph attribute (fallback to vox)
    diam_unit = None
    try:
        if "diameter_unit" in G.attributes():
            diam_unit = G["diameter_unit"]
    except Exception:
        diam_unit = None
    if diam_unit is None:
        # you said PKLs are in vox, so this is the safest default
        print("[WARN] G has no 'diameter_unit'. Assuming diameters/radii are in VOX for density conversion.")
        diam_unit = "vox"

    if diam_unit == "vox":
        sx, sy, sz = map(float, res_um_per_vox)  # e.g. (1.625, 1.625, 2.5)
        scale_r = sx if abs(sx - sy) < 1e-9 else np.sqrt(sx * sy)  # XY-plane scaling
        ms2["r0"] = np.asarray(ms["r0"], float) * scale_r
        ms2["r1"] = np.asarray(ms["r1"], float) * scale_r
    else:
        ms2["r0"] = np.asarray(ms["r0"], float)
        ms2["r1"] = np.asarray(ms["r1"], float)

    return ms2

# ---------------------------------------------------------------------
# COLLECTORS
# ---------------------------------------------------------------------
all_summaries   = []
diam_long_parts = []
path_len_rows   = []
bc_rows         = []
hdn_nodes_rows  = []
red_rows        = []

density_box_values = {}
density_box_slabs  = {}
density_long_rows  = []

density_total_rows = []   # one row per box: total density (no slabs)

# ---------------------------------------------------------------------
# analysis loop (CUT BOXES) — FORMATTED GRAPHS
# ---------------------------------------------------------------------
for (name, path), center in zip(PATHS.items(), HIPPO_CENTERS):
    print("\n======================")
    print("Analyzing:", name)
    print("  ", path)

    G = load_formatted_graph(path)
    G = keep_giant_component(G)

    # ADDED: quick unit sanity print
    g_unit = G["unit"] if ("unit" in G.attributes()) else "unknown"
    d_unit = G["diameter_unit"] if ("diameter_unit" in G.attributes()) else "unknown"
    print(f"[{name}] units: coords/length={g_unit} | diameter={d_unit}")

    # CutBox in um (same convention you used for cutting)
    cut_box = make_box_in_um(center, box_size_um, res_um_per_vox=res_um_per_vox)
    validate_box_faces(cut_box)

    print(
        "BOX bounds:\n"
        f"  x: [{cut_box['xmin']:.6f}, {cut_box['xmax']:.6f}]  (size={cut_box['xmax']-cut_box['xmin']:.6f})\n"
        f"  y: [{cut_box['ymin']:.6f}, {cut_box['ymax']:.6f}]  (size={cut_box['ymax']-cut_box['ymin']:.6f})\n"
        f"  z: [{cut_box['zmin']:.6f}, {cut_box['zmax']:.6f}]  (size={cut_box['zmax']-cut_box['zmin']:.6f})"
    )

    # --- Basic stats
    dup = duplicated_edge_stats(G)
    loops = loop_edge_stats(G)

    # --- Lengths / diameters by type
    avg_len_dict = get_avg_length_nkind(G)
    avg_len_by_type = {EDGE_NKIND_TO_LABEL.get(int(k), str(k)): float(v) for k, v in avg_len_dict.items()}

    diam_stats = diameter_stats_nkind(G, label_dict=EDGE_NKIND_TO_LABEL, plot=True, title_suffix=name)

    # --- per-edge diameter/length (for global overlays)
    # NOTE: if your stored diameters are in vox, then these values are vox (even if column name says _um)
    diam = safe_arr(G.es["diameter"], float)
    leng = safe_arr(G.es["length"], float)
    nk   = safe_arr(G.es["nkind"], int)

    for k in np.unique(nk):
        lab = EDGE_NKIND_TO_LABEL.get(int(k), str(k))
        m = (nk == k)
        if np.any(m):
            diam_long_parts.append(pd.DataFrame({
                "graph": name,
                "type": str(lab),
                "diameter_um": diam[m],  # (still whatever unit diameter is stored in)
                "length_um": leng[m],
            }))

    # per-box length distribution by type
    plot_hist_by_category_general(
        values=safe_arr(G.es["length"], float),
        category=safe_arr(G.es["nkind"], int),
        label_dict=EDGE_NKIND_TO_LABEL,
        bins=bins_hist,
        layout="horizontal",
        density=True,
        show_mean=True,
        variable_name="Edge length (µm)",
        category_name="Vessel type",
        main_title=f"Length distribution by vessel type | {name}"
    )

    # --- HDN analysis
    get_degrees(G, threshold=degree_thr)

    hdn = analyze_hdn_pattern_in_box(
        G,
        box=cut_box,
        coords_attr="coords",
        space="um",
        degree_thr=degree_thr,
        eps_vox=eps_vox
    )

    deg = safe_arr(G.degree(), int)
    hdn_nodes = np.where(deg >= degree_thr)[0]
    coords = np.asarray(G.vs["coords"], float)

    for v in hdn_nodes:
        t = infer_node_type_from_incident_edges(G, int(v))
        x, y, z = coords[int(v)]
        hdn_nodes_rows.append({
            "graph": name,
            "v": int(v),
            "x": float(x), "y": float(y), "z": float(z),
            "degree": int(deg[int(v)]),
            "type": t
        })

    plot_degree_nodes_spatial(
        G,
        coords_attr="coords",
        degree_min=degree_thr,
        degree_max=None,
        by_type=True,
        title=f"High-degree nodes (deg ≥ {degree_thr}) | {name}"
    )

    # --- BC faces (cut graphs) -> use border mode
    bc = analyze_bc_faces(
        G,
        cut_box,
        coords_attr="coords",
        space="um",
        eps_vox=eps_vox,
        degree_thr=degree_thr,
        mode="border"
    )

    debug_face_plane_counts(G, cut_box, coords_attr="coords", eps_vox=eps_vox, space="um")

    bc_df = bc_faces_table(bc, box_name=name).copy()
    bc_df["graph"] = name

    for vname, pcol in {
        "arteriole": "% Arteriole",
        "venule": "% Venule",
        "capillary": "% Capillary",
        "unknown": "% Unknown",
    }.items():
        bc_df[f"n_{vname}"] = (
            pd.to_numeric(bc_df["BC nodes"], errors="coerce").fillna(0.0) *
            pd.to_numeric(bc_df[pcol], errors="coerce").fillna(0.0) / 100.0
        )

    bc_rows.append(bc_df)

    plot_bc_cube_net(bc, title=f"BC composition per face (cube net) | {name}")
    plot_bc_3_cubes_tinted(G, cut_box, coords_attr="coords", space="um", eps_vox=eps_vox, mode="border")

    # --- Redundancy: ALL shortest A->V paths
    shortest_paths = shortest_av_paths(G)

    for p in shortest_paths:
        path_len_rows.append({"graph": name, "path_len_edges": int(len(p) - 1)})

    if shortest_paths:
        plot_av_paths_in_box(
            G,
            cut_box,
            shortest_paths[:plot_paths_cap],
            coords_attr="coords",
            node_eps=0.0
        )

    # -----------------------------------------------------------------
    # Vessel density in CutBox
    # IMPORTANT: convert r0/r1 (vox) -> µm ONLY for density computations
    # -----------------------------------------------------------------
    density_mean_cutbox = np.nan          # mean of slab values (%) depends on slab thickness
    density_total_cutbox = np.nan         # ONE number for whole CutBox (%) between box comparison

    ms_cut = microsegments_from_formatted_graph(G)

    # Use toolbox spacing if available; fallback hard-coded
    try:
        _res = res_um_per_vox
    except NameError:
        _res = (1.625, 1.625, 2.5)

    ms_den = microsegments_for_density_um(ms_cut, G, _res)

    # (B) TOTAL density
    tot = vessel_vol_frac_total_in_box(ms_den, cut_box)
    density_total_cutbox = float(tot["total_vol_frac"]) * 100.0  # %

    density_total_rows.append({
        "graph": name,
        "total_vol_frac_pct": density_total_cutbox,
        "vessel_vol": float(tot.get("vessel_vol", np.nan)),
        "tissue_vol": float(tot.get("tissue_vol", np.nan)),
        "arteriole_vol_frac_pct": float(tot.get("arteriole_vol_frac", np.nan)) * 100.0 if "arteriole_vol_frac" in tot else np.nan,
        "venule_vol_frac_pct": float(tot.get("venule_vol_frac", np.nan)) * 100.0 if "venule_vol_frac" in tot else np.nan,
        "capillary_vol_frac_pct": float(tot.get("capillary_vol_frac", np.nan)) * 100.0 if "capillary_vol_frac" in tot else np.nan,
    })

    # (A) slabs profile
    df_box = vessel_vol_frac_slabs_in_box(ms_den, cut_box, slab=slab_um_cutbox, axis=slab_axis)
    density_box_slabs[name] = df_box.copy()

    y_box = df_box["total_vol_frac"].to_numpy(float) * 100.0  # %
    density_box_values[name] = y_box
    density_mean_cutbox = float(np.nanmean(y_box))

    for v in y_box:
        density_long_rows.append({"graph": name, "slab_vol_frac_pct": float(v)})

    df_box.to_csv(os.path.join(out_root, f"{name}_CutBox_density_slabs_{int(slab_um_cutbox)}um.csv"), index=False)

    try:
        plot_intra_box_heterogeneity(df_box, name, slab_um_cutbox)
    except Exception as e:
        print(f"[{name}] plot intra-box failed (A):", e)

    # --- Distance to surface
    d2s_mean = np.nan
    d2s_median = np.nan
    if ("distance_to_surface_R" in G.vs.attributes()) or ("distance_to_surface" in G.vs.attributes()):
        nodes = np.arange(G.vcount())
        d2s = distance_to_surface_stats(G, nodes, space="um")
        d2s_mean = float(d2s["mean"])
        d2s_median = float(d2s["median"])

    # --- Edge-disjoint paths (maxflow)
    red = max_edge_disjoint_av(G)
    n_disjoint = int(red["n_edge_disjoint_av"])
    nA = int(red["nA"])
    nV = int(red["nV"])

    red_rows.append({
        "graph": name,
        "edge_disjoint_AV": n_disjoint,
        "A_nodes": nA,
        "V_nodes": nV,
        "shortest_paths_n": int(len(shortest_paths)),
        "shortest_path_len_median": float(np.median([len(p)-1 for p in shortest_paths])) if shortest_paths else np.nan,
        "shortest_path_len_p90": float(np.percentile([len(p)-1 for p in shortest_paths], 90)) if shortest_paths else np.nan,
    })

    # --- Summary row (NOW includes ratio fields)
    summary = {
        "graph": name,
        "V": int(G.vcount()),
        "E": int(G.ecount()),
        "dup_%": float(dup["perc_extra_edges"]),
        "loops_%": float(loops["perc_loops"]),
        "HDN_n": int(hdn.get("n_hdn", 0)),
        "HDN_frac": float(hdn.get("hdn_fraction", 0.0)),
        "shortest_paths_n": int(len(shortest_paths)),
        "edge_disjoint_AV": int(n_disjoint),
        "A_nodes": int(nA),
        "V_nodes": int(nV),

        # slab heterogeneity summary (mean across slabs)
        "density_mean_cutbox_%": float(density_mean_cutbox) if np.isfinite(density_mean_cutbox) else np.nan,

        # one-number CutBox density (between boxes)
        "density_total_cutbox_%": float(density_total_cutbox) if np.isfinite(density_total_cutbox) else np.nan,

        "d2s_mean": float(d2s_mean) if np.isfinite(d2s_mean) else np.nan,
        "d2s_median": float(d2s_median) if np.isfinite(d2s_median) else np.nan,

    }

    # avg length by type
    for nm, val in avg_len_by_type.items():
        summary[f"avg_len_{nm}_um"] = float(val)

    # diameter stats by type
    for _, st in diam_stats.items():
        nm = st["name"]
        summary[f"diam_mean_{nm}_proxy_um"]   = float(st["mean"])
        summary[f"diam_median_{nm}_proxy_um"] = float(st["median"])
        summary[f"diam_p5_{nm}_proxy_um"]     = float(st["p5"])
        summary[f"diam_p95_{nm}_proxy_um"]    = float(st["p95"])

    all_summaries.append(summary)

    # save per-box CSVs
    pd.DataFrame([summary]).to_csv(os.path.join(out_root, f"{name}_summary.csv"), index=False)
    bc_df.to_csv(os.path.join(out_root, f"{name}_bc_faces.csv"), index=False)

# ---------------------------------------------------------------------
# Store summary
# ---------------------------------------------------------------------
summary_df = pd.DataFrame(all_summaries)
summary_csv = os.path.join(out_root, "SMC_summary_COMPARISON.csv")
summary_df.to_csv(summary_csv, index=False)
print("\nSaved:", summary_csv)


# =====================================================================
# GLOBAL FIGURES (kept)
# =====================================================================
BOX_LABEL = " vs ".join(BOX_ORDER)

# Diameter/Length distributions per type
if diam_long_parts:
    dl = pd.concat(diam_long_parts, ignore_index=True)
    dl["type"] = dl["type"].astype(str)

    diameter_length_overlay_by_type(dl, bins=bins_hist, box_label=BOX_LABEL)

    for t in sorted(dl["type"].unique()):
        sub = dl[dl["type"] == t].copy()
        plot_boxplot_by_graph(sub, "diameter_um", title=f"Diameter boxplot | type={t}", ylabel="Diameter (µm)")
        plot_boxplot_by_graph(sub, "length_um", title=f"Length boxplot | type={t}", ylabel="Length (µm)")

# CutBox density heterogeneity (10 µm) -> boxplot per CutBox
if density_box_values:
    order = [g for g in BOX_ORDER if g in density_box_values]
    groups = [safe_finite(density_box_values[g]) for g in order]

    plt.figure(figsize=(7, 5))
    plt.boxplot(groups, labels=order, showfliers=False)
    plt.title(f"CutBox vessel density heterogeneity | slab={int(slab_um_cutbox)} µm | {BOX_LABEL}")
    plt.ylabel("Vessel volume fraction per slab (%)")
    plt.grid(alpha=0.25, axis="y")
    plt.tight_layout()
    plt.show()

    pd.DataFrame(density_long_rows).to_csv(
        os.path.join(out_root, f"CutBox_density_long_{int(slab_um_cutbox)}um.csv"),
        index=False
    )

# Shortest path length distribution (#edges): overlay hist + boxplot
if path_len_rows:
    pldf = pd.DataFrame(path_len_rows)

    plot_hist_overlay(
        pldf, "path_len_edges",
        title=f"Shortest A→V path length (#edges) | {BOX_LABEL}",
        xlabel="Path length (#edges)",
        bins=30,
    )
    plot_boxplot_by_graph(
        pldf, "path_len_edges",
        title=f"Shortest A→V path length (#edges) | {BOX_LABEL}",
        ylabel="Path length (#edges)",
    )

print("\nGLOBAL figures generated in:", out_root)

# ---------------------------------------------------------------------
# CutBox TOTAL density between boxes (one value per box)
# ---------------------------------------------------------------------
if density_total_rows:
    df_total = pd.DataFrame(density_total_rows).copy()

    df_total["total_vol_frac_pct"] = pd.to_numeric(df_total["total_vol_frac_pct"], errors="coerce")

    mx = float(np.nanmax(df_total["total_vol_frac_pct"].to_numpy(float))) if len(df_total) else np.nan
    if np.isfinite(mx) and mx <= 1.0:
        print("[TOTAL density] Detected fraction (0–1). Converting to percent (*100).")
        df_total["total_vol_frac_pct"] *= 100.0

    df_total = df_total.groupby("graph", as_index=False).mean(numeric_only=True)

    graphs_in_data = set(df_total["graph"].astype(str))
    order = [g for g in BOX_ORDER if g in graphs_in_data]
    extras = [g for g in df_total["graph"].astype(str).tolist() if g not in BOX_ORDER]

    missing = [g for g in BOX_ORDER if g not in graphs_in_data]
    if missing:
        print("[TOTAL density] WARNING: BOX_ORDER entries missing in data:", missing)
    if extras:
        print("[TOTAL density] NOTE: graphs in data not in BOX_ORDER:", extras)

    df_total = df_total.set_index("graph").loc[order + extras].reset_index()

    total_csv = os.path.join(out_root, "CutBox_density_TOTAL_between_boxes.csv")
    df_total.to_csv(total_csv, index=False)
    print("Saved:", total_csv)
    print(df_total)

    labels = df_total["graph"].astype(str).to_numpy()
    vals = df_total["total_vol_frac_pct"].to_numpy(float)
    m = np.isfinite(vals)

    plt.figure(figsize=(6.5, 4.5))
    plt.bar(labels[m], vals[m])
    plt.title("CutBox vessel density (TOTAL) — between boxes")
    plt.ylabel("Vessel volume fraction (%)")
    plt.grid(alpha=0.25, axis="y")

    for x, v in zip(labels[m], vals[m]):
        plt.text(x, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("[TOTAL density] density_total_rows is empty -> nothing to plot/save.")

# ---------------------------------------------------------------------
# BC cube-net compare (per face, per box, stacked by vessel type)
# ---------------------------------------------------------------------
if len(bc_rows) == 0:
    print("[BC] bc_rows is empty -> nothing to plot.")
else:
    bc_all = pd.concat(bc_rows, ignore_index=True)

    face_col = "Face" if "Face" in bc_all.columns else None
    if face_col is None:
        print("[BC] Could not find face column ('Face') in bc_all.")
    else:
        COLMAP = {
            "arteriole": "n_arteriole",
            "venule": "n_venule",
            "capillary": "n_capillary",
            "unknown": "n_unknown",
        }
        total_col = "BC nodes" if "BC nodes" in bc_all.columns else None

        for col in list(COLMAP.values()) + ([total_col] if total_col else []):
            if col not in bc_all.columns:
                bc_all[col] = 0.0
            bc_all[col] = pd.to_numeric(bc_all[col], errors="coerce").fillna(0.0)

        fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=True)
        fig.suptitle(f"Boundary conditions | cube-net compare | {BOX_LABEL}", fontsize=14)

        for r in range(2):
            for c in range(3):
                face = FACE_GRID[r][c]
                ax = axes[r, c]
                sub = bc_all.loc[bc_all[face_col] == face].copy()

                if sub.empty:
                    ax.set_title(face)
                    ax.text(0.5, 0.5, "no data", ha="center", va="center")
                    ax.set_xticks([])
                    ax.set_yticks([])
                    continue

                group_cols = list(COLMAP.values()) + ([total_col] if total_col else [])
                sub = sub.groupby("graph", as_index=False)[group_cols].sum()

                x = np.arange(len(BOX_ORDER))
                bottom = np.zeros(len(BOX_ORDER), float)

                for vname in ["arteriole", "venule", "capillary", "unknown"]:
                    col = COLMAP[vname]
                    vals = np.array([
                        float(sub.loc[sub["graph"] == g, col].iloc[0]) if (sub["graph"] == g).any() else 0.0
                        for g in BOX_ORDER
                    ], dtype=float)

                    ax.bar(x, vals, bottom=bottom, label=vname, color=V_COL.get(vname, "lightgrey"))
                    bottom += vals

                if total_col and total_col in sub.columns:
                    tot = np.array([
                        float(sub.loc[sub["graph"] == g, total_col].iloc[0]) if (sub["graph"] == g).any() else 0.0
                        for g in BOX_ORDER
                    ], dtype=float)
                    ax.plot(x, tot, "k.", markersize=8, label="total" if (r == 0 and c == 0) else None)

                ax.set_title(face)
                ax.set_xticks(x)
                ax.set_xticklabels(BOX_ORDER)
                ax.grid(alpha=0.25, axis="y")

        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=5, frameon=False)
        plt.tight_layout(rect=[0, 0.06, 1, 0.95])
        plt.show()

# ---------------------------------------------------------------------
# Redundancy: shortest path length ECDF + medians
# ---------------------------------------------------------------------
if path_len_rows:
    sp = pd.DataFrame(path_len_rows)

    plt.figure(figsize=(7.5, 5))
    for g in BOX_ORDER:
        x = sp.loc[sp["graph"] == g, "path_len_edges"].to_numpy(float)
        x = x[np.isfinite(x)]
        if not x.size:
            continue
        xs, ys = ecdf_xy(x)
        plt.plot(xs, ys, label=g, color=BOX_COL.get(g, None))
        plt.axvline(float(np.median(xs)), color=BOX_COL.get(g, None), linestyle="--", alpha=0.8)

    plt.title(f"Shortest A→V path length (ECDF) | {BOX_LABEL}")
    plt.xlabel("Path length (#edges)")
    plt.ylabel("ECDF")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------------
# Edge-disjoint (maxflow) as bars
# ---------------------------------------------------------------------
if red_rows:
    ed = pd.DataFrame(red_rows).set_index("graph").reindex(BOX_ORDER)

    plt.figure(figsize=(6.5, 4.5))
    plt.bar(BOX_ORDER, ed["edge_disjoint_AV"].to_numpy(float))
    plt.title(f"Edge-disjoint A→V paths (maxflow) | {BOX_LABEL}")
    plt.ylabel("# edge-disjoint paths")
    plt.grid(alpha=0.25, axis="y")
    plt.tight_layout()
    plt.show()
    
if diam_long_parts:
    dl = pd.concat(diam_long_parts, ignore_index=True)
    dl["type"] = dl["type"].astype(str)
    plot_grouped_boxplot_types_per_graph(
        dl,
        value_col="diameter_um",
        type_col="type",
        graphs_order=BOX_ORDER,
        types_order=("arteriole","venule","capillary"),
        type_colors=V_COL,
        title=f"Diameter grouped boxplots (3 vessel types per SMC) | {BOX_LABEL}",
        ylabel="Diameter (stored units)"
    )